# Decklist representation experiments

This notebook explores how much useful card representation can be extracted from **decklists alone** before bringing in replay/gameplay data.

It deliberately tests several related but distinct signals:

1. **Weighted PPMI + SVD** — broad deck-context similarity.
2. **NMF archetypes** — interpretable latent deck factors.
3. **Residual interactions** — pair relationships stronger than broad context predicts.
4. **Direct association lift** — unusually strong pairwise relationships with support shown explicitly.
5. **Matched-context conditional lift** — compare drafts containing a card with otherwise-similar drafts that do not contain it.
6. **Leave-one-card-out reconstruction** — test whether deck context can recover a held-out card.

The shared multiplicity weighting uses the probability of seeing a card, or pair of cards, in **10 cards from an assumed 40-card deck**. The aim here is exploration, not a final production implementation.

In [1]:
from pathlib import Path
import math
import warnings

import duckdb
import numpy as np
import pandas as pd

from scipy.sparse import coo_matrix, csr_matrix
from scipy.special import gammaln

from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Works whether the notebook is run from the repo root or notebooks/.
for candidate in [Path("data/17lands.duckdb"), Path("../data/17lands.duckdb")]:
    if candidate.exists():
        DB_PATH = candidate
        break
else:
    raise FileNotFoundError("Could not find data/17lands.duckdb")

DECK_SIZE = 40
CARDS_SEEN = 10
MIN_PAIR_PROB = 1e-5
BROAD_COMPONENTS = 12
EMBEDDING_COMPONENTS = 64
RANDOM_STATE = 42

EXCLUDE_BASICS = True
BASIC_NAMES = {"Plains", "Island", "Swamp", "Mountain", "Forest", "Wastes"}

print("Database:", DB_PATH)

Database: ../data/17lands.duckdb


## 1. Load the decklist data

Two representations are loaded:

- **Build rows** retain actual deck builds.
- **Draft-card maxima** collapse multiple builds in one draft by taking the maximum maindeck copy count for each card. This matches the exploratory draft-level convention used so far: a card used in any build during a draft contributes once, at its maximum copy count.

The exact pairwise cooccurrence calculation later is stricter: two cards only receive pair weight from a build in which they were simultaneously present.

In [2]:
basic_filter = ""
if EXCLUDE_BASICS:
    quoted = ", ".join("'" + name.replace("'", "''") + "'" for name in sorted(BASIC_NAMES))
    basic_filter = f"AND c.card_name NOT IN ({quoted})"

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    build_cards = con.execute(f"""
        SELECT
            d.expansion,
            d.event_type,
            db.draft_id,
            db.build_id,
            db.build_index,
            dbc.card_id,
            c.card_name,
            dbc.deck_count
        FROM deck_builds db
        JOIN drafts d
            ON d.draft_id = db.draft_id
        JOIN deck_build_cards dbc
            ON dbc.build_id = db.build_id
        JOIN cards c
            ON c.card_id = dbc.card_id
        WHERE dbc.deck_count > 0
        {basic_filter}
    """).df()

print(f"Build-card rows: {len(build_cards):,}")
print(f"Drafts: {build_cards['draft_id'].nunique():,}")
print(f"Builds: {build_cards['build_id'].nunique():,}")
print(f"Cards: {build_cards['card_id'].nunique():,}")

display(
    build_cards.groupby(["expansion", "event_type"], dropna=False)
    .agg(drafts=("draft_id", "nunique"), builds=("build_id", "nunique"), cards=("card_id", "nunique"))
    .sort_values("drafts", ascending=False)
)

Build-card rows: 148,008
Drafts: 4,127
Builds: 4,805
Cards: 539


,,drafts,builds,cards
expansion,event_type,,,
Cube - Powered,PremierDraft,4127,4805,539


In [3]:
def log_choose(n, k):
    """log(C(n, k)); returns -inf for impossible combinations."""
    n = np.asarray(n, dtype=float)
    valid = (n >= k) & (k >= 0)
    out = np.full(np.broadcast(n, k).shape, -np.inf, dtype=float)
    out[valid] = gammaln(n[valid] + 1) - gammaln(k + 1) - gammaln(n[valid] - k + 1)
    return out


def p_seen_from_count(counts, deck_size=DECK_SIZE, cards_seen=CARDS_SEEN):
    """Probability of seeing at least one copy in cards_seen cards."""
    counts = np.asarray(counts, dtype=float)
    remaining = deck_size - counts
    denom = gammaln(deck_size + 1) - gammaln(cards_seen + 1) - gammaln(deck_size - cards_seen + 1)
    p_miss = np.where(
        remaining >= cards_seen,
        np.exp(
            gammaln(remaining + 1)
            - gammaln(cards_seen + 1)
            - gammaln(remaining - cards_seen + 1)
            - denom
        ),
        0.0,
    )
    return 1.0 - p_miss


draft_card = (
    build_cards.groupby(["draft_id", "card_id", "card_name"], as_index=False)
    .agg(max_copies=("deck_count", "max"))
)
draft_card["p_seen"] = p_seen_from_count(draft_card["max_copies"].to_numpy())

card_table = (
    draft_card[["card_id", "card_name"]]
    .drop_duplicates()
    .sort_values("card_id")
    .reset_index(drop=True)
)
card_ids = card_table["card_id"].tolist()
card_to_idx = {card_id: i for i, card_id in enumerate(card_ids)}
idx_to_card = {i: card_id for card_id, i in card_to_idx.items()}
card_names = dict(zip(card_table["card_id"], card_table["card_name"]))
name_to_id = {name: card_id for card_id, name in card_names.items()}

draft_ids = sorted(draft_card["draft_id"].unique())
draft_to_idx = {draft_id: i for i, draft_id in enumerate(draft_ids)}

rows = draft_card["draft_id"].map(draft_to_idx).to_numpy()
cols = draft_card["card_id"].map(card_to_idx).to_numpy()
weighted = draft_card["p_seen"].to_numpy(dtype=float)
presence = np.ones(len(draft_card), dtype=float)

X_draft = coo_matrix((weighted, (rows, cols)), shape=(len(draft_ids), len(card_ids))).tocsr()
X_presence = coo_matrix((presence, (rows, cols)), shape=X_draft.shape).tocsr()

print("Draft x card matrix:", X_draft.shape)
print("Non-zero entries:", X_draft.nnz)

Draft x card matrix: (4127, 539)
Non-zero entries: 127840


## 2. Exact weighted PPMI

For a build containing (a) copies of card A and (b) copies of card B, the pair contribution is the exact probability of seeing at least one of each in 10 cards from a 40-card deck:

$
P(A\cap B)=1-P(\neg A)-P(\neg B)+P(\neg A\cap\neg B)
$

Within each draft, the maximum simultaneous pair probability across actual builds is used. PMI then asks whether the pair is seen together more often than their marginal frequencies predict.

In [4]:
pair_query = f"""
WITH
build_cards AS (
    SELECT
        db.draft_id,
        db.build_id,
        dbc.card_id,
        dbc.deck_count
    FROM deck_builds db
    JOIN deck_build_cards dbc
        ON dbc.build_id = db.build_id
    JOIN cards c
        ON c.card_id = dbc.card_id
    WHERE dbc.deck_count > 0
    {basic_filter}
),
card_build_prob AS (
    SELECT
        draft_id,
        build_id,
        card_id,
        CASE
            WHEN {DECK_SIZE} - deck_count >= {CARDS_SEEN}
            THEN 1.0 - EXP(
                LGAMMA({DECK_SIZE} - deck_count + 1)
                - LGAMMA({CARDS_SEEN} + 1)
                - LGAMMA({DECK_SIZE} - deck_count - {CARDS_SEEN} + 1)
                - (
                    LGAMMA({DECK_SIZE} + 1)
                    - LGAMMA({CARDS_SEEN} + 1)
                    - LGAMMA({DECK_SIZE} - {CARDS_SEEN} + 1)
                )
            )
            ELSE 1.0
        END AS p_seen
    FROM build_cards
),
build_pair_prob AS (
    SELECT
        a.draft_id,
        a.build_id,
        a.card_id AS card_a_id,
        b.card_id AS card_b_id,
        1.0
        - CASE WHEN {DECK_SIZE} - a.deck_count >= {CARDS_SEEN} THEN EXP(
            LGAMMA({DECK_SIZE} - a.deck_count + 1)
            - LGAMMA({CARDS_SEEN} + 1)
            - LGAMMA({DECK_SIZE} - a.deck_count - {CARDS_SEEN} + 1)
            - (LGAMMA({DECK_SIZE} + 1) - LGAMMA({CARDS_SEEN} + 1) - LGAMMA({DECK_SIZE} - {CARDS_SEEN} + 1))
          ) ELSE 0.0 END
        - CASE WHEN {DECK_SIZE} - b.deck_count >= {CARDS_SEEN} THEN EXP(
            LGAMMA({DECK_SIZE} - b.deck_count + 1)
            - LGAMMA({CARDS_SEEN} + 1)
            - LGAMMA({DECK_SIZE} - b.deck_count - {CARDS_SEEN} + 1)
            - (LGAMMA({DECK_SIZE} + 1) - LGAMMA({CARDS_SEEN} + 1) - LGAMMA({DECK_SIZE} - {CARDS_SEEN} + 1))
          ) ELSE 0.0 END
        + CASE WHEN {DECK_SIZE} - a.deck_count - b.deck_count >= {CARDS_SEEN} THEN EXP(
            LGAMMA({DECK_SIZE} - a.deck_count - b.deck_count + 1)
            - LGAMMA({CARDS_SEEN} + 1)
            - LGAMMA({DECK_SIZE} - a.deck_count - b.deck_count - {CARDS_SEEN} + 1)
            - (LGAMMA({DECK_SIZE} + 1) - LGAMMA({CARDS_SEEN} + 1) - LGAMMA({DECK_SIZE} - {CARDS_SEEN} + 1))
          ) ELSE 0.0 END AS p_seen_together
    FROM build_cards a
    JOIN build_cards b
        ON a.build_id = b.build_id
       AND a.card_id < b.card_id
),
draft_card_prob AS (
    SELECT draft_id, card_id, MAX(p_seen) AS p_seen
    FROM card_build_prob
    GROUP BY draft_id, card_id
),
draft_pair_prob AS (
    SELECT draft_id, card_a_id, card_b_id, MAX(p_seen_together) AS p_seen_together
    FROM build_pair_prob
    GROUP BY draft_id, card_a_id, card_b_id
),
draft_count AS (
    SELECT COUNT(DISTINCT draft_id)::DOUBLE AS n
    FROM deck_builds
),
card_prob AS (
    SELECT dcp.card_id, SUM(dcp.p_seen) / dc.n AS p_card
    FROM draft_card_prob dcp
    CROSS JOIN draft_count dc
    GROUP BY dcp.card_id, dc.n
),
pair_prob AS (
    SELECT dpp.card_a_id, dpp.card_b_id, SUM(dpp.p_seen_together) / dc.n AS p_ab
    FROM draft_pair_prob dpp
    CROSS JOIN draft_count dc
    GROUP BY dpp.card_a_id, dpp.card_b_id, dc.n
)
SELECT
    pp.card_a_id,
    pp.card_b_id,
    ca.card_name AS card_a,
    cb.card_name AS card_b,
    pp.p_ab,
    ap.p_card AS p_a,
    bp.p_card AS p_b,
    LN(pp.p_ab / (ap.p_card * bp.p_card)) AS pmi
FROM pair_prob pp
JOIN card_prob ap ON ap.card_id = pp.card_a_id
JOIN card_prob bp ON bp.card_id = pp.card_b_id
JOIN cards ca ON ca.card_id = pp.card_a_id
JOIN cards cb ON cb.card_id = pp.card_b_id
WHERE pp.p_ab > ?
  AND ap.p_card > 0
  AND bp.p_card > 0
"""

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    pairs = con.execute(pair_query, [MIN_PAIR_PROB]).df()

pairs["ppmi"] = pairs["pmi"].clip(lower=0)
pairs = pairs[np.isfinite(pairs["ppmi"]) & (pairs["ppmi"] > 0)].copy()

ppmi_rows, ppmi_cols, ppmi_values = [], [], []
for row in pairs.itertuples():
    if row.card_a_id not in card_to_idx or row.card_b_id not in card_to_idx:
        continue
    i = card_to_idx[row.card_a_id]
    j = card_to_idx[row.card_b_id]
    ppmi_rows.extend([i, j])
    ppmi_cols.extend([j, i])
    ppmi_values.extend([row.ppmi, row.ppmi])

ppmi_matrix = coo_matrix(
    (ppmi_values, (ppmi_rows, ppmi_cols)),
    shape=(len(card_ids), len(card_ids)),
).tocsr()

print(f"Positive-PMI pairs: {len(pairs):,}")
print(f"Symmetric PPMI non-zeros: {ppmi_matrix.nnz:,}")

Positive-PMI pairs: 48,979
Symmetric PPMI non-zeros: 97,958


## 3. Baseline: broad PPMI/SVD embedding

This is intentionally the simple baseline. It mostly answers:

> Do these cards live in similar deck environments?

That is useful, but it is expected to contain strong colour/archetype structure.

In [5]:
svd = TruncatedSVD(
    n_components=min(EMBEDDING_COMPONENTS, ppmi_matrix.shape[1] - 1),
    random_state=RANDOM_STATE,
)
embeddings = normalize(svd.fit_transform(ppmi_matrix))

print(f"Explained variance: {svd.explained_variance_ratio_.sum():.3f}")


def similar_cards(card_name, topn=15, vectors=embeddings):
    card_id = name_to_id[card_name]
    idx = card_to_idx[card_id]
    sims = vectors @ vectors[idx]
    order = np.argsort(-sims)
    out = []
    for other_idx in order:
        if other_idx == idx:
            continue
        out.append({"card": card_names[idx_to_card[other_idx]], "similarity": float(sims[other_idx])})
        if len(out) == topn:
            break
    return pd.DataFrame(out)


for example in ["Lightning Bolt", "Underworld Breach"]:
    if example in name_to_id:
        print("\n", example)
        display(similar_cards(example, 12))

Explained variance: 0.911

 Lightning Bolt


,card,similarity
0,Broadside Bombardiers,0.992132
1,"Magda, Brazen Outlaw",0.991443
2,Burst Lightning,0.987507
3,Fury,0.987377
4,Chain Lightning,0.986700
5,Generous Plunderer,0.986232
6,Bonecrusher Giant,0.984782
7,"Ragavan, Nimble Pilferer",0.984750
8,"Inti, Seneschal of the Sun",0.982263
9,"Laelia, the Blade Reforged",0.981999



 Underworld Breach


,card,similarity
0,Brain Freeze,0.974808
1,Lion's Eye Diamond,0.965132
2,Wheel of Fortune,0.892964
3,Lotus Petal,0.889724
4,Sleight of Hand,0.877975
5,Yawgmoth's Will,0.876590
6,Echo of Eons,0.870864
7,Tendrils of Agony,0.860452
8,Mystical Tutor,0.853408
9,Thundering Falls,0.852105


## 4. NMF archetypes

NMF factorizes the non-negative draft × card matrix into non-negative latent factors. These components are often easier to interpret than SVD dimensions and give a direct diagnostic of whether decklists contain recognizable archetypes.

This is not trying to remove archetype information; it is explicitly measuring it.

In [6]:
N_ARCHETYPES = min(BROAD_COMPONENTS, max(2, X_draft.shape[1] - 1))

nmf_decks = NMF(
    n_components=N_ARCHETYPES,
    init="nndsvda",
    random_state=RANDOM_STATE,
    max_iter=600,
)

draft_factors = nmf_decks.fit_transform(X_draft)
card_archetype_loadings = nmf_decks.components_.T


def show_archetypes(top_cards=12):
    tables = []
    for k in range(N_ARCHETYPES):
        order = np.argsort(-nmf_decks.components_[k])[:top_cards]
        tables.append(pd.DataFrame({
            "component": k,
            "card": [card_names[idx_to_card[i]] for i in order],
            "loading": nmf_decks.components_[k, order],
        }))
    return pd.concat(tables, ignore_index=True)

archetypes = show_archetypes(12)
display(archetypes.pivot(index="card", columns="component", values="loading").fillna(0).style.format("{:.3f}"))

component,0,1,2,3,4,5,6,7,8,9,10,11
card,,,,,,,,,,,,
"Adeline, Resplendent Cathar",0.000,0.642,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
"Ajani, Nacatl Pariah",0.000,0.602,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Animate Dead,0.000,0.000,1.122,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Archon of Cruelty,0.000,0.000,0.956,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Arid Mesa,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.828
"Atraxa, Grand Unifier",0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.596,0.000,0.000,0.000,0.000
Badlands,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.531,0.000,0.000,0.000
Balance,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.729,0.000,0.000
Barrowgoyf,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.606,0.000,0.000,0.000


## 5. Residual interaction structure

Now use NMF on the **PPMI card × card matrix** to model broad non-negative card context. For every observed PPMI edge:

$
residual(A,B)=PPMI(A,B)-\widehat{PPMI}(A,B)
$

Positive residual means A and B associate more strongly than the broad latent structure predicts. This is the experiment most directly aimed at finding card-specific packages such as **Underworld Breach → Brain Freeze / Lion's Eye Diamond** rather than merely another card from the same colour pair.

In [7]:
nmf_context = NMF(
    n_components=min(BROAD_COMPONENTS, max(2, ppmi_matrix.shape[1] - 1)),
    init="nndsvda",
    random_state=RANDOM_STATE,
    max_iter=800,
)

context_W = nmf_context.fit_transform(ppmi_matrix)
context_H = nmf_context.components_

obs = ppmi_matrix.tocoo()
predicted = np.sum(context_W[obs.row] * context_H[:, obs.col].T, axis=1)
residual_values = np.maximum(obs.data - predicted, 0.0)
keep = residual_values > 0

residual_matrix = coo_matrix(
    (residual_values[keep], (obs.row[keep], obs.col[keep])),
    shape=ppmi_matrix.shape,
).tocsr()

residual_svd = TruncatedSVD(
    n_components=min(EMBEDDING_COMPONENTS, residual_matrix.shape[1] - 1),
    random_state=RANDOM_STATE,
)
residual_embeddings = normalize(residual_svd.fit_transform(residual_matrix))

print(f"Original PPMI non-zeros: {ppmi_matrix.nnz:,}")
print(f"Positive residual non-zeros: {residual_matrix.nnz:,}")
print(f"Residual SVD variance explained: {residual_svd.explained_variance_ratio_.sum():.3f}")


def unusual_partners(card_name, topn=20):
    card_id = name_to_id[card_name]
    i = card_to_idx[card_id]

    row = ppmi_matrix.getrow(i)
    results = []
    for j, observed in zip(row.indices, row.data):
        expected = float(context_W[i] @ context_H[:, j])
        residual = observed - expected
        if residual > 0:
            results.append({
                "other_card": card_names[idx_to_card[j]],
                "ppmi": float(observed),
                "expected": expected,
                "residual": float(residual),
            })

    return pd.DataFrame(results).sort_values("residual", ascending=False).head(topn).reset_index(drop=True)


for example in ["Underworld Breach", "Lightning Bolt", "Chain Lightning"]:
    if example in name_to_id:
        print("\n", example)
        display(unusual_partners(example, 15))

Original PPMI non-zeros: 97,958
Positive residual non-zeros: 59,463
Residual SVD variance explained: 0.649

 Underworld Breach


,other_card,ppmi,expected,residual
0,Regrowth,0.927166,0.374253,0.552913
1,Enlightened Tutor,1.510751,1.008121,0.502630
2,Brain Freeze,2.235600,1.744524,0.491076
3,Pyrite Spellbomb,1.095346,0.629306,0.466040
4,Lion's Eye Diamond,2.128242,1.707758,0.420484
5,Galvanic Blast,0.656213,0.335967,0.320247
6,Spirebluff Canal,0.918054,0.601007,0.317047
7,Soul-Guide Lantern,0.713918,0.405965,0.307953
8,Proft's Eidetic Memory,0.599866,0.296396,0.303470
9,Pinnacle Emissary,0.611971,0.332251,0.279720



 Lightning Bolt


,other_card,ppmi,expected,residual
0,"Kroxa, Titan of Death's Hunger",0.643792,0.320465,0.323327
1,Sunbaked Canyon,0.753423,0.464167,0.289256
2,Bloodtithe Harvester,0.511484,0.225548,0.285936
3,Blackcleave Cliffs,0.392878,0.121500,0.271378
4,Unlicensed Hearse,0.332535,0.073247,0.259289
5,Questing Druid,0.458374,0.199887,0.258487
6,Grim Lavamancer,0.931526,0.673063,0.258463
7,Copperline Gorge,0.336404,0.094007,0.242396
8,Fiery Islet,0.304748,0.071054,0.233694
9,Volcanic Island,0.266550,0.036216,0.230334



 Chain Lightning


,other_card,ppmi,expected,residual
0,Bloodbraid Elf,0.609959,0.204290,0.405669
1,Ademi of the Silkchutes,0.870008,0.506421,0.363587
2,Reprieve,0.456341,0.137956,0.318386
3,Questing Druid,0.505991,0.230937,0.275054
4,Smuggler's Copter,0.668800,0.402900,0.265900
5,Wooded Foothills,0.345090,0.099463,0.245626
6,Fiery Confluence,0.802487,0.572915,0.229572
7,"Loot, the Pathfinder",0.260781,0.046509,0.214272
8,Stomping Ground,0.310247,0.106906,0.203341
9,"Magda, Brazen Outlaw",0.850156,0.647163,0.202992


## 6. Direct association lift

PMI is already a log-lift quantity, so exponentiating it gives an intuitive ratio:

$
lift(A,B)=\frac{P(A,B)}{P(A)P(B)}=e^{PMI(A,B)}
$

A lift of 3 means the weighted pair appears about three times as often as independence would predict. Always inspect support (p_ab) alongside lift; very rare pairs can have extreme PMI.

In [8]:
pairs["lift"] = np.exp(pairs["pmi"])


def association_partners(card_name, topn=20, min_joint_probability=5e-5):
    card_id = name_to_id[card_name]
    x = pairs[
        ((pairs["card_a_id"] == card_id) | (pairs["card_b_id"] == card_id))
        & (pairs["p_ab"] >= min_joint_probability)
    ].copy()
    x["other_card"] = np.where(x["card_a_id"] == card_id, x["card_b"], x["card_a"])
    return (
        x[["other_card", "p_ab", "pmi", "lift"]]
        .sort_values(["lift", "p_ab"], ascending=False)
        .head(topn)
        .reset_index(drop=True)
    )


if "Underworld Breach" in name_to_id:
    display(association_partners("Underworld Breach", 20))

,other_card,p_ab,pmi,lift
0,Brain Freeze,0.002181,2.235600,9.352091
1,Lion's Eye Diamond,0.001775,2.128242,8.400087
2,Tendrils of Agony,0.000685,2.121518,8.343795
3,Seething Song,0.000336,1.925695,6.859913
4,Yawgmoth's Will,0.000587,1.620313,5.054673
5,Wishclaw Talisman,0.000461,1.551001,4.716190
6,Enlightened Tutor,0.000419,1.510751,4.530131
7,Manamorphose,0.000489,1.437992,4.212227
8,"Jace, Wielder of Mysteries",0.000461,1.401966,4.063179
9,Mystical Tutor,0.000839,1.173605,3.233629


## 7. Matched-context conditional lift

This test tries to answer the more causal-looking question:

> Among otherwise-similar drafts, what cards are more common when card A is present?

It is **not causal inference** — draft choices are highly confounded — but it is a useful decklist-only diagnostic.

Procedure:

1. Build a broad low-dimensional representation of each draft.
2. Temporarily remove the target card from the draft matrix.
3. For drafts containing A, find nearest drafts without A.
4. Compare card prevalence in the treated drafts with prevalence in their matched controls.

This explicitly asks whether A changes the surrounding card distribution beyond broad shell similarity.

In [9]:
deck_context_svd = TruncatedSVD(
    n_components=min(32, X_draft.shape[1] - 1),
    random_state=RANDOM_STATE,
)
deck_context_svd.fit(X_draft)


def matched_context_lift(
    card_name,
    topn=20,
    n_neighbors=5,
    max_treated=500,
    min_treated_rate=0.02,
):
    card_id = name_to_id[card_name]
    card_idx = card_to_idx[card_id]

    target_present = np.asarray(X_presence[:, card_idx].todense()).ravel() > 0
    treated_idx = np.flatnonzero(target_present)
    control_idx = np.flatnonzero(~target_present)

    if len(treated_idx) == 0 or len(control_idx) < n_neighbors:
        raise ValueError("Not enough treated/control drafts for matching")

    rng = np.random.default_rng(RANDOM_STATE)
    if len(treated_idx) > max_treated:
        treated_idx = rng.choice(treated_idx, size=max_treated, replace=False)

    # Remove the target card before matching so it cannot identify treated drafts directly.
    X_context = X_draft.copy().tolil()
    X_context[:, card_idx] = 0.0
    X_context = X_context.tocsr()
    context_vectors = normalize(deck_context_svd.transform(X_context))

    nn = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
    nn.fit(context_vectors[control_idx])
    _, neighbor_pos = nn.kneighbors(context_vectors[treated_idx])
    matched_control_idx = control_idx[neighbor_pos]

    treated_rate = np.asarray(X_presence[treated_idx].mean(axis=0)).ravel()

    # Keep duplicate matched controls because repeated matches are part of the matching weight.
    matched_rows = matched_control_idx.ravel()
    matched_rate = np.asarray(X_presence[matched_rows].mean(axis=0)).ravel()

    delta = treated_rate - matched_rate
    ratio = (treated_rate + 1e-6) / (matched_rate + 1e-6)

    result = pd.DataFrame({
        "card_id": card_ids,
        "card": [card_names[c] for c in card_ids],
        "with_target": treated_rate,
        "matched_without_target": matched_rate,
        "delta": delta,
        "ratio": ratio,
    })

    result = result[
        (result["card_id"] != card_id)
        & (result["with_target"] >= min_treated_rate)
    ]

    return result.sort_values(["delta", "ratio"], ascending=False).head(topn).reset_index(drop=True)


for example in ["Underworld Breach", "Lightning Bolt"]:
    if example in name_to_id:
        print("\n", example)
        display(matched_context_lift(example, topn=20))


 Underworld Breach


,card_id,card,with_target,matched_without_target,delta,ratio
0,9088715036539948993,Brain Freeze,0.655462,0.285714,0.369748,2.294113
1,396546654461862736,Lion's Eye Diamond,0.533613,0.276471,0.257143,1.930088
2,4195925205681220158,Enlightened Tutor,0.126050,0.026050,0.100000,4.838562
3,1000727096761612189,Consider,0.163866,0.100000,0.063866,1.638649
4,1172371732768814095,Pyrite Spellbomb,0.142857,0.081513,0.061345,1.752568
5,1025203406798635264,Mana Confluence,0.113445,0.052101,0.061345,2.177397
6,8892917546399089578,Mystical Tutor,0.252101,0.197479,0.054622,1.276594
7,9091775856747995840,Riverpyre Verge,0.184874,0.131933,0.052941,1.401271
8,3407888900320185212,Malevolent Rumble,0.117647,0.065546,0.052101,1.794860
9,1344095259201420366,Raucous Theater,0.138655,0.088235,0.050420,1.571422



 Lightning Bolt


,card_id,card,with_target,matched_without_target,delta,ratio
0,3119212633864723974,"Laelia, the Blade Reforged",0.227991,0.178330,0.049661,1.278479
1,6237367789668717470,Volcanic Island,0.142212,0.093905,0.048307,1.514418
2,5374936410102379276,Fury,0.218962,0.173363,0.045598,1.263019
3,926802840955848862,Sunbaked Canyon,0.187359,0.145824,0.041535,1.284828
4,6197412147731599587,Fiery Islet,0.106095,0.064560,0.041535,1.643347
5,3736919354156838772,Bloodstained Mire,0.137698,0.097065,0.040632,1.418600
6,637243323694041255,Wooded Foothills,0.146727,0.106998,0.039729,1.371305
7,7783959816338929574,Carnage Interpreter,0.124153,0.084876,0.039278,1.462761
8,3104031594665373674,Pyrogoyf,0.214447,0.176524,0.037923,1.214833
9,3650021965498144011,Monastery Swiftspear,0.088036,0.053725,0.034312,1.638644


## 8. Leave-one-card-out deck reconstruction

This is a practical evaluation rather than another similarity display.

Take a real build, hide one card, and rank candidate cards using only the remaining deck. The notebook compares:

- **Popularity baseline** — globally common cards.
- **SVD reconstruction** — broad latent deck structure.
- **NMF reconstruction** — non-negative archetype structure.
- **PPMI context score** — direct positive association with the remaining cards.

This does not prove a representation is semantically correct, but it tells us whether it contains enough deck information to recover plausible omitted cards.

In [10]:
# Use one actual build per draft so drafts with many build changes do not dominate evaluation.
representative_builds = (
    build_cards[["expansion", "event_type", "draft_id", "build_id", "build_index"]]
    .drop_duplicates()
    .sort_values(["draft_id", "build_index", "build_id"])
    .drop_duplicates("draft_id")
)

rep_rows = build_cards.merge(
    representative_builds[["draft_id", "build_id"]],
    on=["draft_id", "build_id"],
    how="inner",
)

build_ids = representative_builds["build_id"].tolist()
build_to_idx = {build_id: i for i, build_id in enumerate(build_ids)}

r = rep_rows["build_id"].map(build_to_idx).to_numpy()
c = rep_rows["card_id"].map(card_to_idx).to_numpy()
v = p_seen_from_count(rep_rows["deck_count"].to_numpy())
X_build = coo_matrix((v, (r, c)), shape=(len(build_ids), len(card_ids))).tocsr()
X_build_presence = (X_build > 0).astype(float).tocsr()

build_svd = TruncatedSVD(
    n_components=min(64, X_build.shape[1] - 1),
    random_state=RANDOM_STATE,
)
build_svd.fit(X_build)

build_nmf = NMF(
    n_components=min(BROAD_COMPONENTS, X_build.shape[1] - 1),
    init="nndsvda",
    random_state=RANDOM_STATE,
    max_iter=600,
)
build_nmf.fit(X_build)

popularity = np.asarray(X_build_presence.mean(axis=0)).ravel()


def rank_of(scores, heldout_idx, forbidden):
    scores = np.asarray(scores, dtype=float).copy()
    scores[list(forbidden)] = -np.inf
    target_score = scores[heldout_idx]
    return int(1 + np.sum(scores > target_score))


def evaluate_leave_one_out(n_examples=250, min_cards=8):
    rng = np.random.default_rng(RANDOM_STATE)
    eligible = np.array([i for i in range(X_build.shape[0]) if X_build_presence[i].nnz >= min_cards])
    if len(eligible) > n_examples:
        eligible = rng.choice(eligible, size=n_examples, replace=False)

    records = []

    for row_idx in eligible:
        present = X_build_presence[row_idx].indices
        heldout = int(rng.choice(present))

        x = X_build[row_idx].copy().tolil()
        x[0, heldout] = 0.0
        x = x.tocsr()

        forbidden = set(present) - {heldout}

        # Broad low-rank reconstruction.
        svd_scores = build_svd.inverse_transform(build_svd.transform(x))[0]

        # Non-negative archetype reconstruction.
        nmf_scores = build_nmf.transform(x) @ build_nmf.components_
        nmf_scores = nmf_scores[0]

        # Direct PPMI compatibility with the remaining cards.
        remaining = list(forbidden)
        if remaining:
            ppmi_scores = np.asarray(ppmi_matrix[:, remaining].sum(axis=1)).ravel() / len(remaining)
        else:
            ppmi_scores = np.zeros(len(card_ids))

        methods = {
            "popularity": popularity,
            "svd": svd_scores,
            "nmf": nmf_scores,
            "ppmi": ppmi_scores,
        }

        for method, scores in methods.items():
            rank = rank_of(scores, heldout, forbidden)
            records.append({
                "method": method,
                "rank": rank,
                "reciprocal_rank": 1.0 / rank,
                "hit_10": rank <= 10,
                "hit_20": rank <= 20,
            })

    raw = pd.DataFrame(records)
    summary = raw.groupby("method").agg(
        mean_reciprocal_rank=("reciprocal_rank", "mean"),
        hit_at_10=("hit_10", "mean"),
        hit_at_20=("hit_20", "mean"),
        median_rank=("rank", "median"),
        examples=("rank", "size"),
    ).sort_values("mean_reciprocal_rank", ascending=False)

    return summary, raw

loo_summary, loo_raw = evaluate_leave_one_out()
display(loo_summary)

,mean_reciprocal_rank,hit_at_10,hit_at_20,median_rank,examples
method,,,,,
svd,0.109501,0.232,0.352,37.0,250
ppmi,0.062151,0.144,0.252,57.0,250
nmf,0.056047,0.156,0.236,58.0,250
popularity,0.028397,0.052,0.072,179.5,250


## 9. Hard-negative reconstruction

The ordinary leave-one-out benchmark can be won by learning broad archetype. This stricter diagnostic gives each held-out card a candidate set made of cards with **similar NMF archetype loadings**.

Within that deliberately difficult candidate set, compare direct PPMI context with residual context. If residual structure helps here, it is evidence that decklists contain signal beyond broad archetype membership.

In [11]:
archetype_card_vectors = normalize(card_archetype_loadings)


def evaluate_hard_negatives(n_examples=200, candidate_pool=50, min_cards=8):
    rng = np.random.default_rng(RANDOM_STATE)
    eligible = np.array([
        i for i in range(X_build.shape[0])
        if X_build_presence[i].nnz >= min_cards
    ])

    if len(eligible) > n_examples:
        eligible = rng.choice(eligible, size=n_examples, replace=False)

    records = []

    for row_idx in eligible:
        present = X_build_presence[row_idx].indices
        heldout = int(rng.choice(present))
        remaining = [i for i in present if i != heldout]

        if not remaining:
            continue

        # Build a hard candidate pool from cards with similar broad archetype loadings.
        broad_sims = archetype_card_vectors @ archetype_card_vectors[heldout]
        broad_order = np.argsort(-broad_sims)

        candidates = [heldout]
        for idx in broad_order:
            if idx == heldout or idx in present:
                continue
            candidates.append(int(idx))
            if len(candidates) >= candidate_pool:
                break

        candidates = np.array(candidates, dtype=int)

        ppmi_scores = (
            np.asarray(ppmi_matrix[candidates][:, remaining].sum(axis=1)).ravel()
            / len(remaining)
        )
        residual_scores = (
            np.asarray(residual_matrix[candidates][:, remaining].sum(axis=1)).ravel()
            / len(remaining)
        )

        for method, scores in {
            "ppmi": ppmi_scores,
            "residual": residual_scores,
        }.items():
            target_score = scores[0]
            rank = int(1 + np.sum(scores[1:] > target_score))
            records.append({
                "method": method,
                "rank": rank,
                "reciprocal_rank": 1.0 / rank,
                "hit_5": rank <= 5,
                "hit_10": rank <= 10,
            })

    raw = pd.DataFrame(records)
    summary = raw.groupby("method").agg(
        mean_reciprocal_rank=("reciprocal_rank", "mean"),
        hit_at_5=("hit_5", "mean"),
        hit_at_10=("hit_10", "mean"),
        median_rank=("rank", "median"),
        examples=("rank", "size"),
    ).sort_values("mean_reciprocal_rank", ascending=False)

    return summary, raw


hard_negative_summary, hard_negative_raw = evaluate_hard_negatives()
display(hard_negative_summary)

,mean_reciprocal_rank,hit_at_5,hit_at_10,median_rank,examples
method,,,,,
residual,0.166266,0.245,0.440,13.0,200
ppmi,0.161561,0.225,0.395,14.0,200


## 10. Compare several views of one card

This final helper puts the exploratory signals next to one another. A useful card representation probably should not collapse all of these into the same notion of similarity:

- broad context tells us **where the card belongs**;
- residuals tell us **what is unusually associated with it**;
- matched lift tells us **what changes in otherwise-similar decks when it appears**.

Look for cases where these tell coherent but different stories.

In [12]:
def card_report(card_name, topn=12):
    if card_name not in name_to_id:
        raise KeyError(card_name)

    print(f"=== {card_name}: broad PPMI/SVD neighbours ===")
    display(similar_cards(card_name, topn))

    print(f"=== {card_name}: unusual residual partners ===")
    display(unusual_partners(card_name, topn))

    print(f"=== {card_name}: direct association lift ===")
    display(association_partners(card_name, topn))

    print(f"=== {card_name}: matched-context lift ===")
    display(matched_context_lift(card_name, topn=topn))


for example in ["Underworld Breach", "Lightning Bolt", "Faerie Mastermind"]:
    if example in name_to_id:
        card_report(example, 10)

=== Underworld Breach: broad PPMI/SVD neighbours ===


,card,similarity
0,Brain Freeze,0.974808
1,Lion's Eye Diamond,0.965132
2,Wheel of Fortune,0.892964
3,Lotus Petal,0.889724
4,Sleight of Hand,0.877975
5,Yawgmoth's Will,0.876590
6,Echo of Eons,0.870864
7,Tendrils of Agony,0.860452
8,Mystical Tutor,0.853408
9,Thundering Falls,0.852105


=== Underworld Breach: unusual residual partners ===


,other_card,ppmi,expected,residual
0,Regrowth,0.927166,0.374253,0.552913
1,Enlightened Tutor,1.510751,1.008121,0.502630
2,Brain Freeze,2.235600,1.744524,0.491076
3,Pyrite Spellbomb,1.095346,0.629306,0.466040
4,Lion's Eye Diamond,2.128242,1.707758,0.420484
5,Galvanic Blast,0.656213,0.335967,0.320247
6,Spirebluff Canal,0.918054,0.601007,0.317047
7,Soul-Guide Lantern,0.713918,0.405965,0.307953
8,Proft's Eidetic Memory,0.599866,0.296396,0.303470
9,Pinnacle Emissary,0.611971,0.332251,0.279720


=== Underworld Breach: direct association lift ===


...

,other_card,ppmi,expected,residual
0,Up the Beanstalk,0.659475,0.106715,0.552759
1,Duelist of the Mind,1.280301,0.789538,0.490763
2,Enduring Curiosity,1.292403,0.805789,0.486614
3,Dark Confidant,0.487005,0.065040,0.421966
4,Proft's Eidetic Memory,1.194398,0.775231,0.419166
5,Dauthi Voidwalker,0.508336,0.090707,0.417628
6,Field of the Dead,0.453622,0.048471,0.405152
7,Mox Jet,0.432202,0.044301,0.387901
8,Mox Sapphire,0.426722,0.077927,0.348795
9,Brazen Borrower,1.154915,0.817210,0.337705


=== Faerie Mastermind: direct association lift ===


,other_card,p_ab,pmi,lift
0,Enduring Curiosity,0.000405,1.292403,3.641526
1,Duelist of the Mind,0.000336,1.280301,3.597723
2,Proft's Eidetic Memory,0.000294,1.194398,3.301569
3,Brazen Borrower,0.000601,1.154915,3.173754
4,Cryptic Command,0.000489,0.995947,2.707286
5,Phantasmal Image,0.000252,0.992619,2.698292
6,Treasure Cruise,0.000391,0.985996,2.680481
7,Fallen Shinobi,0.000489,0.964448,2.623339
8,Abhorrent Oculus,0.000322,0.915770,2.498699
9,Quantum Riddler,0.000475,0.857208,2.356573


=== Faerie Mastermind: matched-context lift ===


,card_id,card,with_target,matched_without_target,delta,ratio
0,4854103978404181996,Brazen Borrower,0.218274,0.140102,0.078173,1.557967
1,9112820776835652341,Mox Sapphire,0.147208,0.076142,0.071066,1.933321
2,637243323694041255,Wooded Foothills,0.121827,0.059898,0.061929,2.033881
3,6742113973125308957,Duelist of the Mind,0.121827,0.065990,0.055838,1.846141
4,5480932988245545101,Mox Jet,0.147208,0.093401,0.053807,1.576081
5,2529918906750272522,Enduring Curiosity,0.147208,0.095431,0.051777,1.542548
6,852767726285313779,Aether Spellbomb,0.116751,0.065990,0.050761,1.769219
7,1293341248286063436,Scalding Tarn,0.172589,0.121827,0.050761,1.416663
8,2301071416006240625,Blood Crypt,0.101523,0.053807,0.047716,1.886776
9,213839180250705395,Hullbreacher,0.223350,0.179695,0.043655,1.242937


## What to look for

This notebook is useful even if the final model looks completely different.

**Promising evidence from decklists** would be:

- NMF factors that correspond to recognizable archetypes rather than arbitrary popularity buckets.
- Residual partners that reveal specific packages or synergies after broad context is removed.
- Matched-context lift that finds sensible differences between decks containing a card and similar decks without it.
- Leave-one-out reconstruction that beats the popularity baseline by a meaningful margin.

**Warning signs** would be:

- every method mostly returning the same colour/archetype lists;
- residual rankings dominated by tiny-support accidents;
- matched lift being unstable when the neighbour count or sample changes;
- leave-one-out performance barely beating popularity.

If these tests show both broad archetype structure and genuinely card-specific residual structure, then decklists are doing something useful: not providing the complete card representation, but providing multiple complementary signals that can later be combined with gameplay/replay representations.